# Prompt engineering to Agentic RAG 🤖🧠
### LangChain RAG Agent — Prompt Engineering with a Hugging Face Dataset
---

> This notebook builds a Retrieval-Augmented Generation agent with LangChain and the public AG News dataset from Hugging Face.
> The goal is to keep the same retrieval pipeline while changing the system prompt to compare grounding, citations, format, length, tone, audience adaptation, clarification behavior, and 
> insufficient-context handling.
> AG News is a historical benchmark dataset. Treat it as a news archive, not a live news service.

## Learning objectives

1. Load a public Hugging Face dataset.
2. Convert rows into LangChain documents.
3. Build an in-memory vector store.
4. Expose retrieval as an agent tool.
5. Compare weak and optimized prompts.
6. Test ambiguity and insufficient-context handling.
7. Evaluate output quality.

**Install dependencies**

In [ ]:
%pip install -U datasets langchain langchain-openai langchain-community langchain-text-splitters python-dotenv colorama

**Environment setup**

In [ ]:
import os
from getpass import getpass
from colorama import Fore, Style
from dotenv import load_dotenv

load_dotenv(override=True)

if not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('Enter your OpenAI API key: ')
else:
    print(f'{Fore.GREEN}OpenAI API key loaded.{Style.RESET_ALL}')
    
    
if not os.getenv('LANGSMITH_API_KEY'):
    os.environ['LANGSMITH_API_KEY'] = getpass('Enter your LANGSMITH API key: ')
else:
    print(f'{Fore.GREEN}LANGSMITH API key loaded.{Style.RESET_ALL}')
    
    
print(os.getenv("LANGSMITH_TRACING"))
print(os.getenv("LANGSMITH_API_KEY"))
print(os.getenv("LANGSMITH_PROJECT"))


**Initialize the model**

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    'openai:gpt-4.1-mini',
    temperature=0,
    timeout=30,
    max_retries=3,
)

print('Model initialized.')

**Define prompt variations**
variations of prompts with diff instructions, model's role, in tone, style 
generate and compare diff outputs
so we can see the progression from a weak to a strong and well-designed prompt:

In [ ]:
WEAK_PROMPT = '''You are a news assistant.
Use the search tool and answer the user's question.'''

GROUNDED_PROMPT = '''You are a grounded news-archive assistant.
The knowledge base is a historical AG News dataset, not a real-time news service.
Use search_news_archive before answering news questions.
Use only retrieved evidence. Cite factual claims using [Source N].
If evidence is insufficient, say: "The available archive does not contain enough information."
Begin with a direct sentence, add three concise bullets with category and citation, finish with one limitation, and use no more than 180 words.'''

BEGINNER_FRIENDLY_PROMPT = '''You are a teacher explaining historical news to learners with little business or technology knowledge.
Always use search_news_archive. Use only retrieved evidence.
Start with a plain-English explanation, explain difficult terms, use at most three bullets, include one example, cite factual claims using [Source N], end with "In simple terms, ...", and write 140 to 180 words.'''

EXECUTIVE_PROMPT = '''You are preparing a short briefing for a business leader.
Use search_news_archive and rely only on retrieved articles.
Return exactly: ## Headline, ## Key Signals with exactly three cited bullets, ## Business Relevance, and ## Data Limitation.
Use direct, decision-oriented language and a maximum of 140 words.'''

CLARIFYING_PROMPT = '''You are a grounded news-archive assistant.
If the request is ambiguous, ask one concise clarification question and do not search yet.
If it is specific, use search_news_archive, answer only from retrieved evidence, cite [Source N], and state when evidence is insufficient.'''

ANALYTICAL_PROMPT = '''You are an analytical research assistant working with a historical news archive.
Use search_news_archive before answering.
Return exactly: ## Main Finding, ## Evidence with three cited bullets, ## Interpretation, and ## Limitation.
Use only retrieved evidence, neutral language, and no more than 220 words.'''

**Load the AG News dataset**

Using public dataset from Hugging Face: [AG News](https://huggingface.co/datasets/fancyzhx/ag_news)
A small slice keeps the demo fast and inexpensive.

In [ ]:
from datasets import load_dataset

dataset = load_dataset('fancyzhx/ag_news', split='train[:500]')
print(dataset)
print(dataset[0])

**Convert data into LangChain documents**

In [ ]:
from langchain_core.documents import Document

LABELS = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Science and Technology'}

documents = []
for index, row in enumerate(dataset):
    label = int(row['label'])
    documents.append(
        Document(
            page_content=row['text'].strip(),
            metadata={
                'source': f'ag_news_{index}',
                'category': LABELS[label],
                'dataset': 'AG News',
                'record_id': index,
            },
        )
    )

print(f'Documents created: {len(documents)}')
print(documents[0])

**Split, embed, and index the documents**

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vector_store = InMemoryVectorStore(embedding=embeddings)
vector_store.add_documents(chunks)
retriever = vector_store.as_retriever(search_kwargs={'k': 5})

print(f'Documents: {len(documents)}')
print(f'Chunks: {len(chunks)}')

**Format retrieved documents**

In [ ]:
def format_documents(docs: list[Document]) -> str:
    if not docs:
        return 'No relevant articles were retrieved.'

    formatted = []
    for position, doc in enumerate(docs, start=1):
        formatted.append(
            f'[Source {position}]\n'
            f"Record: {doc.metadata.get('source', 'unknown')}\n"
            f"Category: {doc.metadata.get('category', 'unknown')}\n"
            f'Article: {doc.page_content}'
        )
    return '\n\n'.join(formatted)

**Create the retrieval tool**

In [ ]:
from langchain.tools import tool

@tool
def search_news_archive(query: str) -> str:
    """Search the historical AG News archive for relevant articles."""
    retrieved_documents = retriever.invoke(query)
    if not retrieved_documents:
        return 'STATUS: NO_RESULTS\nNo relevant news articles were found.'
    return 'STATUS: RESULTS_FOUND\n\n' + format_documents(retrieved_documents)

**Create the agents**
compare results with a variations of prompts.


In [ ]:
from langchain.agents import create_agent

agents = {
    'Weak': create_agent(model=model, tools=[search_news_archive], system_prompt=WEAK_PROMPT),
    'Grounded': create_agent(model=model, tools=[search_news_archive], system_prompt=GROUNDED_PROMPT),
    'Beginner': create_agent(model=model, tools=[search_news_archive], system_prompt=BEGINNER_FRIENDLY_PROMPT),
    'Executive': create_agent(model=model, tools=[search_news_archive], system_prompt=EXECUTIVE_PROMPT),
    'Clarifying': create_agent(model=model, tools=[search_news_archive], system_prompt=CLARIFYING_PROMPT),
    'Analytical': create_agent(model=model, tools=[search_news_archive], system_prompt=ANALYTICAL_PROMPT),
}
print(list(agents))

**Helper to invoke an agent**

In [ ]:
from langsmith import traceable


@traceable(name="RAG Agent")
def ask_agent(agent, question: str) -> str:
    result = agent.invoke({'messages': [{'role': 'user', 'content': question}]})
    return result['messages'][-1].content[:180]

**Compare outputs for the same question**

In [ ]:
question = 'What do the articles say about technology companies?'

for name in ['Weak', 'Grounded', 'Beginner', 'Executive', 'Analytical']:
    print('\n' + '=' * 80)
    print(name.upper())
    print('=' * 80)
    print(ask_agent(agents[name], question))

**Ambiguous-query demo**

In [ ]:
ambiguous_question = 'What happened with Apple?'
print(ask_agent(agents['Clarifying'], ambiguous_question))

**Insufficient-context demo**

In [ ]:
unsupported_question = 'What did Apple announce yesterday, and how did its share price react?'
print(ask_agent(agents['Grounded'], unsupported_question))

**Inspect retrieval directly**

## Key takeaway

Prompt engineering by itsel does not improve the underlying evidence. It improves how the agent retrieves, interprets, structures, and communicates that evidence. Strong prompts cannot compensate for poor retrieval, but they can make a well-retrieved answer more reliable, useful, and audience-appropriate.